## API query - conversion to points
### 1. Import libraries

This notebook demonstrates how to query information from a web API and convert it to a spatial layer in ArcGIS Pro.

- **requests**: Handles HTTP requests. Allows us to retrieve data from the web.
- **pandas**: Used to manage and manipulate the data returned from the API
- **arcpy**: Works with spatial data in ArcGIS Pro

In [ ]:
import requests
import pandas as pd
import arcpy

### 2. Define the API endpoint

An API endpoint is a URL within an API that allows you to access a particular resource or perform a specific action.
Here's a breakdown of the key components:

- **Base URL**: The main address of the API. For example, https://api.example.com.

- **Endpoint Path**: The specific path added to the base URL to access a resource. For example, /data in https://api.example.com/data.

- **URL Parameters**: Additional parameters added to the endpoint to filter or modify the request. These are typically included in the query string of the URL and follow a key=value format. The documentation for the specific API you're working with will usually explain what parameters are available and what they do. For example, in https://api.example.com/data?type=user&limit=10, type=user and limit=10 are URL parameters.

- **API Key**: When you sign up for access to an API, the provider typically generates an API key for you. It helps track usage and enforce rate limits for security and management purposes. You will usually need to add your API Key as a URL parameter.

For instance, in the URL https://api.openbrewerydb.org/v1/breweries, https://api.openbrewerydb.org/v1 is the base URL, and /breweries is the endpoint path. If you send a request to this endpoint, it will return a list of breweries.

In [ ]:
# Define the API endpoint. This API doesn't require an API key. You can learn about the available URL parameters here: https://openbrewerydb.org/documentation/
url = 'https://api.openbrewerydb.org/v1/breweries?by_country=united%20states&by_state=wisconsin'

## 3. Retrieve the data and convert it to a DataFrame
First, the requests library will be used to get the data. We will check if the request was successful by looking at the response status code. If a '200' is returned, that means the request was successful. The data will be returned in JSON format, which we will convert to a Python dictionary. Then we'll convert the dictionary to a pandas DataFrame.

After the results are converted to a dataframe, the dataframe is printed to the screen. Note the 'longitude' and 'latitude' columns. These will be passed to the XY Table to Point tool in the next step.

In [ ]:
# Send a GET request to the API
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    # Parse the JSON response
    data = response.json()
    
    # Convert the JSON data to a pandas DataFrame
    df = pd.DataFrame(data)
    
    print(df)
else:
    print(f"Failed to retrieve data: {response.status_code}")

## 4. Save the data as a CSV and add it to the map
We will use the arcpy module to get our project's home folder. Then we'll save that dataframe as a csv in the home folder. The XY Table to Point geoprocessing tool, provided by arcpy, will be used to transform the csv into a spatial layer in the map. Note that this method only returns the first 50 results. To get the complete set of results, pagination must be taken into account (see notebook #2).

In [ ]:
# Create a variable to refer to the current ArcGIS Pro project
aprx = arcpy.mp.ArcGISProject('CURRENT')

# Save the results of the API query to a csv (spreadsheet file)
# in your project's home folder.
output_csv = aprx.homeFolder + '\\' + 'query_results.csv'
df.to_csv(output_csv)

# Use the XY Table to Point geoprocessing tool (https://pro.arcgis.com/en/pro-app/latest/tool-reference/data-management/xy-table-to-point.htm)
# to turn the CSV data into a spatial layer.
arcpy.management.XYTableToPoint(output_csv, 'results8', 'longitude', 'latitude') 